<a href="https://colab.research.google.com/github/kunphat510214-netizen/Final-Final/blob/step-7-8/Coffee_Shop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ขั้นที่ 7: วิเคราะห์ข้อมูลด้วย pandas และ SQL

ก่อนรัน ให้วาง `coffee_orders.csv` และ `coffee_shop.db` ไว้ในโฟลเดอร์เดียวกับโน้ตบุ๊ก หรืออัปโหลดเข้า Colab ก่อน


In [1]:
# @title
"""ขั้นที่ 7: วิเคราะห์ข้อมูลร้านกาแฟด้วย pandas และ SQL.""" # คำอธิบายภาพรวมของกระบวนการวิเคราะห์

from pathlib import Path # นำเข้าไลบรารีจัดการเส้นทางไฟล์
import sqlite3 # นำเข้าไลบรารีสำหรับจัดการฐานข้อมูล SQL

import pandas as pd # นำเข้าไลบรารี pandas สำหรับวิเคราะห์ข้อมูล


CSV_PATH = Path("coffee_orders.csv") # กำหนดตำแหน่งไฟล์ CSV
DATABASE_PATH = Path("coffee_shop.db") # กำหนดตำแหน่งไฟล์ฐานข้อมูล


def show(title, dataframe): # ฟังก์ชันสำหรับแสดงผลตารางข้อมูล
    """แสดง DataFrame ให้อ่านได้ง่าย.""" # คำอธิบายการทำงานของฟังก์ชัน
    print(f"\n{title}") # พิมพ์ชื่อหัวข้อการวิเคราะห์
    print(dataframe.to_string(index=False)) # พิมพ์ข้อมูลโดยไม่แสดงเลขลำดับแถว


if not CSV_PATH.exists(): # ตรวจสอบการมีอยู่ของไฟล์ CSV
    raise FileNotFoundError("ไม่พบ coffee_orders.csv กรุณารันขั้นที่ 5-6 ก่อน") # แจ้งข้อผิดพลาดหากไม่พบไฟล์
if not DATABASE_PATH.exists(): # ตรวจสอบการมีอยู่ของไฟล์ฐานข้อมูล
    raise FileNotFoundError("ไม่พบ coffee_shop.db กรุณารันขั้นที่ 5-6 ก่อน") # แจ้งข้อผิดพลาดหากไม่พบฐานข้อมูล


# 7.1 โหลด CSV และสำรวจข้อมูลเบื้องต้น # เริ่มขั้นตอนโหลดและสำรวจข้อมูล
coffee_df = pd.read_csv(CSV_PATH) # โหลดข้อมูลจาก CSV เข้าสู่ DataFrame
coffee_df.info() # แสดงรายละเอียดโครงสร้างข้อมูลเบื้องต้น
show("สถิติเบื้องต้น", coffee_df.describe(include="all").reset_index()) # แสดงค่าสถิติพื้นฐานของข้อมูลทั้งหมด

FileNotFoundError: ไม่พบ coffee_orders.csv กรุณารันขั้นที่ 5-6 ก่อน

In [ ]:
# 7.2 วิเคราะห์ด้วย pandas: groupby + agg + sort_values + Top 5 # ขั้นตอนการวิเคราะห์เชิงลึกด้วย pandas
menu_summary = ( # เริ่มต้นสร้างตัวแปรสรุปข้อมูลเมนู
    coffee_df.groupby("menu_name") # จัดกลุ่มข้อมูลตามชื่อเมนู
    .agg( # คำนวณค่าทางสถิติตามกลุ่ม
        total_orders=("order_id", "count"), # นับจำนวนคำสั่งซื้อรวม
        total_revenue=("price", "sum"), # คำนวณยอดขายรวม
        avg_price=("price", "mean"), # คำนวณราคาเฉลี่ยต่อหน่วย
        avg_wait_minutes=("wait_minutes", "mean"), # คำนวณเวลารอเฉลี่ย
    )
    .reset_index() # แปลงกลุ่มข้อมูลกลับเป็นคอลัมน์ปกติ
    .sort_values("total_revenue", ascending=False) # เรียงลำดับตามรายได้สูงสุดไปต่ำสุด
)
show("Top 5 เมนูที่สร้างรายได้สูงสุด", menu_summary.head(5)) # แสดงผลการวิเคราะห์ 5 อันดับแรก

In [ ]:
# 7.3 วิเคราะห์ด้วย SQL # ขั้นตอนการวิเคราะห์ข้อมูลด้วย SQL Query
queries = { # สร้างชุดคำสั่ง SQL
    "ออเดอร์ราคาสูง": """ -- ค้นหาเมนูที่ราคาสูงกว่า 75 บาท
        SELECT order_id, queue_no, menu_name, price
        FROM orders
        WHERE price >= 75
        ORDER BY price DESC
        LIMIT 5
    """,
    "QR Code และกลับบ้าน": """ -- ค้นหาการสั่งกลับบ้านที่จ่ายด้วย QR Code
        SELECT order_id, menu_name, receive_type, payment_method, price
        FROM orders
        WHERE receive_type = 'กลับบ้าน'
          AND payment_method = 'QR Code'
        ORDER BY order_id
        LIMIT 5
    """,
    "คิวรอนาน": """ -- ค้นหาออเดอร์ที่ใช้เวลารอนานกว่า 15 นาที
        SELECT order_id, menu_name, wait_minutes
        FROM orders
        WHERE wait_minutes >= 15
        ORDER BY wait_minutes DESC, order_id
        LIMIT 5
    """,
    "รายได้ตามช่องทางชำระ": """ -- สรุปรายได้แยกตามวิธีการชำระเงิน
        SELECT payment_method,
               COUNT(*) AS total_orders,
               ROUND(SUM(price), 2) AS total_revenue,
               ROUND(AVG(price), 2) AS avg_price
        FROM orders
        GROUP BY payment_method
        ORDER BY total_revenue DESC
    """,
    "JOIN สมาชิกกับออเดอร์": """ -- เชื่อมข้อมูลการสั่งซื้อกับรายชื่อสมาชิก
        SELECT o.order_id, o.queue_no, m.member_name, o.menu_name, o.price
        FROM orders AS o
        JOIN members AS m ON o.member_id = m.member_id
        WHERE o.price >= 60
        ORDER BY o.price DESC
        LIMIT 5
    """,
} # สิ้นสุดการกำหนด Dictionary

with sqlite3.connect(DATABASE_PATH) as connection: # เปิดการเชื่อมต่อฐานข้อมูล SQLite
    for title, query in queries.items(): # วนลูปประมวลผลคำสั่ง SQL ทีละคำสั่ง
        show(title, pd.read_sql_query(query, connection)) # อ่านผลลัพธ์ SQL และแสดงผลทางหน้าจอ

# ขั้นที่ 8: สร้างกราฟและสรุปผล

ก่อนรัน ให้วาง `coffee_orders.csv` ไว้ในโฟลเดอร์เดียวกับโน้ตบุ๊ก หรืออัปโหลดเข้า Colab ก่อน


In [ ]:
"""ขั้นที่ 8: สร้างกราฟและสรุปผลการวิเคราะห์ร้านกาแฟ."""

from pathlib import Path  # นำเข้าโมดูล Path สำหรับการจัดการพาธไฟล์

import matplotlib.pyplot as plt  # นำเข้า Matplotlib สำหรับการสร้างกราฟ
import pandas as pd  # นำเข้า Pandas สำหรับการจัดการข้อมูลในรูปแบบ DataFrame


CSV_PATH = Path("coffee_orders.csv")  # กำหนดพาธของไฟล์ CSV ที่มีข้อมูลออเดอร์กาแฟ
CHART_PATH = Path("coffee_analysis.png")  # กำหนดพาธสำหรับบันทึกไฟล์กราฟผลการวิเคราะห์

if not CSV_PATH.exists():  # ตรวจสอบว่าไฟล์ coffee_orders.csv มีอยู่หรือไม่
    raise FileNotFoundError("ไม่พบ coffee_orders.csv กรุณารันขั้นที่ 5-6 ก่อน")  # ถ้าไม่พบไฟล์ ให้แสดงข้อผิดพลาด


# โหลดข้อมูลใหม่เพื่อให้ไฟล์ขั้นที่ 8 รันแยกจากขั้นที่ 7 ได้
coffee_df = pd.read_csv(CSV_PATH)  # โหลดข้อมูลจากไฟล์ CSV เข้าสู่ DataFrame

menu_summary = (  # สร้าง DataFrame สรุปข้อมูลเมนู
    coffee_df.groupby("menu_name")  # จัดกลุ่มข้อมูลตามชื่อเมนู
    .agg(  # ใช้ฟังก์ชัน aggregate เพื่อคำนวณค่าต่างๆ
        total_orders=("order_id", "count"),  # นับจำนวนออเดอร์ทั้งหมดของแต่ละเมนู
        total_revenue=("price", "sum"),  # รวมรายได้ทั้งหมดของแต่ละเมนู
        avg_wait_minutes=("wait_minutes", "mean"),  # คำนวณเวลารอเฉลี่ยของแต่ละเมนู
    )
    .reset_index()  # รีเซ็ต index เพื่อให้ 'menu_name' กลับมาเป็นคอลัมน์ปกติ
    .sort_values("total_revenue", ascending=False)  # เรียงลำดับเมนูตามรายได้รวมจากมากไปน้อย
)
top5_menu = menu_summary.head(5)  # เลือก 5 อันดับแรกของเมนูที่มีรายได้สูงสุด
payment_counts = coffee_df["payment_method"].value_counts()  # นับจำนวนออเดอร์ตามวิธีการชำระเงิน

In [ ]:
# 8.1 สร้างกราฟอย่างน้อย 3 กราฟ
fig, axes = plt.subplots(1, 3, figsize=(18, 5))  # สร้างพื้นที่สำหรับกราฟ 3 กราฟในแถวเดียวกัน

top5_menu.plot.bar(  # สร้างกราฟแท่งสำหรับ 5 อันดับเมนู
    x="menu_name",  # กำหนดแกน x เป็นชื่อเมนู
    y="total_revenue",  # กำหนดแกน y เป็นรายได้รวม
    ax=axes[0],  # วางในตำแหน่งกราฟที่ 1
    legend=False,  # ไม่แสดงคำอธิบายสัญลักษณ์
    color="steelblue",  # กำหนดสีแท่งกราฟ
)
axes[0].set_title("Top 5 Menu Revenue")  # ตั้งชื่อหัวข้อกราฟที่ 1
axes[0].set_xlabel("Menu")  # ตั้งชื่อแกน x
axes[0].set_ylabel("Revenue (THB)")  # ตั้งชื่อแกน y

payment_counts.plot.bar(ax=axes[1], color="seagreen")  # สร้างกราฟแท่งจำนวนออเดอร์ตามวิธีชำระเงินในตำแหน่งที่ 2
axes[1].set_title("Orders by Payment Method")  # ตั้งชื่อหัวข้อกราฟที่ 2
axes[1].set_xlabel("Payment Method")  # ตั้งชื่อแกน x
axes[1].set_ylabel("Orders")  # ตั้งชื่อแกน y

coffee_df["wait_minutes"].plot.hist(  # สร้างกราฟฮิสโตแกรมแสดงการกระจายของเวลารอ
    bins=9,  # แบ่งช่วงข้อมูลเป็น 9 ช่วง
    ax=axes[2],  # วางในตำแหน่งกราฟที่ 3
    color="darkorange",  # กำหนดสีส้ม
    edgecolor="black",  # กำหนดสีขอบแท่ง
)
axes[2].set_title("Queue Waiting Time")  # ตั้งชื่อหัวข้อกราฟที่ 3
axes[2].set_xlabel("Minutes")  # ตั้งชื่อแกน x
axes[2].set_ylabel("Orders")  # ตั้งชื่อแกน y

plt.tight_layout()  # ปรับการวางเลย์เอาต์ของกราฟให้เหมาะสมไม่ซ้อนทับกัน
plt.savefig(CHART_PATH, dpi=150, bbox_inches="tight")  # บันทึกรูปภาพกราฟลงไฟล์
plt.show()  # แสดงกราฟออกมาบนหน้าจอ

In [ ]:
# 8.2 สรุปผลอย่างน้อย 3-5 ประโยค
best_menu = menu_summary.iloc[0]  # ดึงข้อมูลเมนูที่อยู่อันดับแรก (รายได้สูงสุด)
member_share = coffee_df["member_id"].notna().mean() * 100  # คำนวณเปอร์เซ็นต์ของลูกค้าที่เป็นสมาชิก
popular_payment = payment_counts.index[0]  # ดึงชื่อวิธีการชำระเงินที่นิยมที่สุด
average_wait = coffee_df["wait_minutes"].mean()  # คำนวณค่าเฉลี่ยของเวลารอคิว

print(  # แสดงผลสรุปข้อที่ 1 เรื่องเมนูรายได้สูงสุด
    f"1. เมนูที่สร้างรายได้สูงสุดคือ {best_menu['menu_name']} "
    f"รวม {best_menu['total_revenue']:,.2f} บาท"
)
print(f"2. ลูกค้าสมาชิกคิดเป็น {member_share:.1f}% ของออเดอร์ทั้งหมด")  # แสดงผลสรุปข้อที่ 2 เรื่องสมาชิก
print(f"3. ช่องทางชำระเงินที่ใช้มากที่สุดคือ {popular_payment}")  # แสดงผลสรุปข้อที่ 3 เรื่องการชำระเงิน
print(f"4. เวลารอเฉลี่ยอยู่ที่ {average_wait:.1f} นาที")  # แสดงผลสรุปข้อที่ 4 เรื่องเวลารอ
print("5. ร้านควรเตรียมวัตถุดิบเมนูยอดนิยมและเพิ่มกำลังคนเมื่อคิวรอนาน")  # แสดงข้อเสนอแนะเพิ่มเติม
print(f"บันทึกกราฟไว้ที่ {CHART_PATH}")  # แจ้งตำแหน่งที่บันทึกไฟล์รูปภาพ